In [71]:
import os
import cv2
import numpy as np
import pandas as pd
from PIL import Image
import plotly.graph_objects as go
from sklearn.decomposition import PCA
from pattern_recognition.data.loadTraces import LoadTraces
from pattern_recognition.data_preprocessing.speqle.concatenate_traces import CurveConcatenator

In [72]:
correct_order = [
    'basAnt', 'basAntSept', 'basSept', 'basInf', 'basPost', 'basLat',
    'midAnt', 'midAntSept', 'midSept', 'midInf', 'midPost', 'midLat',
    'apAnt', 'apAntSept', 'apSept', 'apInf', 'apPost', 'apLat'
]

In [73]:
# pcs_to_plot = [2]
pcs_to_plot = [13, 17, 32] # 0, 1, 2, 3,

# Phillips

In [74]:
phi_trace_loader = LoadTraces('BL', 'phi_segmentalStrainInterp_T', None, isComponents=None)
phi_DF = phi_trace_loader.tracesDF
phi_DF = phi_DF[~phi_DF['curve_segment'].str.lower().str.contains('apex', na=False)]

# GE

In [75]:
ste_trace_loader = LoadTraces('BL', 'st_segmentalStrainInterp_T', None, isComponents=None)
ste_DF = ste_trace_loader.tracesDF

## Concatenate the two DataFrames

In [76]:
contact_st_df = pd.concat([ste_DF, phi_DF], ignore_index=True)
tracesDF = contact_st_df.groupby('study_nr').filter(lambda x: len(x) == 18)

In [77]:
tracesDF['cardiac_view'] = tracesDF['cardiac_view'].str.lower()
tracesDF['segment_wall'] = tracesDF['segment_wall'].str.lower().str.strip()
dropna_tracestracesDF = tracesDF.dropna(subset=['strain_trace'])
dropna_tracestracesDF = dropna_tracestracesDF[~dropna_tracestracesDF['strain_trace'].apply(lambda x: np.all(np.array(x) == 0))]
tracestracesDF = dropna_tracestracesDF.sort_values(by=['study_nr', 'id_position'])
study_nr = np.sort(tracestracesDF['study_nr'].unique())

In [78]:
# study_nr_imputed = tracesDF.loc[tracesDF['isImputed'] == True, 'study_nr'].unique()
# df_isImputed_1 = tracesDF.loc[tracesDF['study_nr'].isin(study_nr_imputed)]
# df_isImputed_0 = tracesDF.loc[~tracesDF['study_nr'].isin(study_nr_imputed)]

In [79]:
concatenated_strain = CurveConcatenator(tracestracesDF, isReconstructed=True)
concat_tracesDF = concatenated_strain.concatenate_traces_par()

Concatenating traces...


In [80]:
# concatenated_strain = CurveConcatenator(df_isImputed_1, isReconstructed=True)
# concat_tracesDF_1 = concatenated_strain.concatenate_traces_par()
#
# concatenated_strain = CurveConcatenator(df_isImputed_0, isReconstructed=True)
# concat_tracesDF_0 = concatenated_strain.concatenate_traces_par()

## Compute mean across rows on train data

In [81]:
stacked_traces = np.stack(concat_tracesDF['concatenated_trace'].values)
mean_trace = np.mean(stacked_traces, axis=0)
print(mean_trace)
len(mean_trace)

[ 0.          0.17604204  0.33685821 ... -0.60824141 -0.26036494
  0.        ]


2304

In [82]:
segment_names = concat_tracesDF['concat_curve_segm'][0].tolist()
# segment_names

In [83]:
pca = PCA(n_components=0.95)
X_transformed = pca.fit_transform(stacked_traces)
n_components = pca.n_components_
print(f"Number of components to retain 95% variance: {n_components}")

Number of components to retain 95% variance: 59


In [84]:
# components_ = pca.components_
explained_variance = pca.explained_variance_ratio_
mean_vector = pca.mean_

# Re order segments

In [85]:
# Reorder segments for visualization
def reorder_trace_and_segments(mean_trace, segment_names, correct_order):
    segment_len = len(mean_trace) // len(segment_names)
    segment_dict = {seg: mean_trace[i * segment_len: (i + 1) * segment_len]
                    for i, seg in enumerate(segment_names)}

    reordered_trace = np.concatenate([segment_dict[seg] for seg in correct_order])
    return reordered_trace, correct_order


mean_trace_reordered, segment_names_reordered = reorder_trace_and_segments(
    mean_trace, segment_names, correct_order
)


In [86]:
# X_test = np.stack(concat_tracesDF_0['concatenated_trace'].values)
# mean_test_trace = np.mean(X_test, axis=0)

# scale components to pc

In [87]:
def effect_vector(pca, pc_index):
    return np.sqrt(pca.explained_variance_[pc_index]) * pca.components_[pc_index]

def reorder_vector_by_segments(vec, original_names, new_order):
    seg_len = len(vec) // len(original_names)
    blocks = {name: vec[i*seg_len:(i+1)*seg_len] for i, name in enumerate(original_names)}
    return np.concatenate([blocks[name] for name in new_order])

def rescale_to_target_norm(vec, target_norm, eps=1e-12):
    n = np.linalg.norm(vec)
    if n < eps:
        return vec
    return vec * (target_norm / n)


# Utilities

In [88]:
def get_scaled_component(pca, pc_index):
    return np.sqrt(pca.explained_variance_[pc_index]) * pca.components_[pc_index]

In [89]:
def get_alpha_loop():
    ascending = np.round(np.arange(0.1, 1.1, 0.1), 2)
    descending = np.round(np.arange(0.9, 0.0, -0.1), 2)
    neg_ascending = np.round(np.arange(-0.1, -1.1, -0.1), 2)
    neg_descending = np.round(np.arange(-0.9, -0.0, 0.1), 2)
    return np.concatenate([ascending, descending, [0], neg_ascending, neg_descending, [0]])

def get_color(alpha):
    if alpha > 0:
        return "royalblue"
    elif alpha < 0:
        return "red"
    return "gray"

def compute_yaxis_range(original, component_scaled, alphas):
    max_offset = np.max(np.abs(alphas)) * np.max(np.abs(component_scaled))
    y_min = np.min(original) - max_offset
    y_max = np.max(original) + max_offset
    margin = 0.1 * (y_max - y_min)
    return [y_min - margin, y_max + margin]


In [90]:
def plot_trace_with_reference(trace, reference, label, color, title, filename, yaxis_range, x_ticks=None, x_labels=None, vlines=None):
    fig = go.Figure()

    fig.add_trace(go.Scatter(
        y=trace, mode="lines", name=label,
        line=dict(color=color, width=2, dash="solid")
    ))

    fig.add_trace(go.Scatter(
        y=reference, mode="lines", name="Original",
        line=dict(color="black", width=2, dash="solid")
    ))

    layout = dict(
        title=title,
        xaxis_title="Timepoints",
        yaxis=dict(
            title="Strain",
            range=yaxis_range,
            showgrid=True,
            zeroline=True
        ),
        template="plotly_white",
        height=500,
        width=800,
        legend=dict(orientation="h", y=-0.2)
    )

    layout["xaxis"] = dict(
    title="Timepoints",
    tickmode="array",
    tickvals=x_ticks,
    ticktext=x_labels,
    tickangle=45,
    showgrid=True,
    zeroline=False
    )

    fig.update_layout(**layout)

    if vlines:
        for x in vlines:
            fig.add_vline(x=x, line=dict(color="gray", dash="dot", width=1))

    fig.write_image(filename)

# Visualize Variations

In [91]:
def create_video_from_frames(frame_paths, video_path, fps=3):
    if not frame_paths:
        raise ValueError("No frames to create video.")

    frame = Image.open(frame_paths[0])
    width, height = frame.size

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    video = cv2.VideoWriter(video_path, fourcc, fps, (width, height))

    for path in frame_paths:
        img = cv2.imread(path)
        video.write(img)

    video.release()

# By segments

In [92]:
def visualize_pca_segmental(pca, mean_trace, segment_names, pc_index,
                            output_dir="segmental_cineloops", fps=4, equalize_to_pc1=True):

    os.makedirs(output_dir, exist_ok=True)
    alphas = get_alpha_loop()
    n_segments = len(segment_names)
    seg_len = len(mean_trace) // n_segments

    # PC effect (reordered) + optional equalization to PC1
    pc_effect = effect_vector(pca, pc_index)
    pc_effect = reorder_vector_by_segments(pc_effect, segment_names, segment_names)

    if equalize_to_pc1:
        pc1_effect = effect_vector(pca, 1)
        pc1_effect = reorder_vector_by_segments(pc1_effect, segment_names, segment_names)
        target_norm = np.linalg.norm(pc1_effect)
        pc_effect = rescale_to_target_norm(pc_effect, target_norm)

    yaxis_range = compute_yaxis_range(mean_trace, pc_effect, alphas)

    for seg_idx, seg_name in enumerate(segment_names):
        print(f"🎥 Segment: {seg_name} — PC {pc_index}")
        start, end = seg_idx * seg_len, (seg_idx + 1) * seg_len
        original_seg = mean_trace[start:end]

        frame_dir = os.path.join(output_dir, f"frames_{seg_name}_pc{pc_index}")
        os.makedirs(frame_dir, exist_ok=True)
        frame_paths = []

        for i, alpha in enumerate(alphas):
            seg_trace = (mean_trace + alpha * pc_effect)[start:end]
            label = f"α={alpha:.2f} × PC{pc_index}"
            filename = os.path.join(frame_dir, f"{seg_name}_pc{pc_index}_frame_{i:03d}.png")

            plot_trace_with_reference(
                trace=seg_trace,
                reference=original_seg,
                label=label,
                color=get_color(alpha),
                title=f"{seg_name} — PC {pc_index}",
                filename=filename,
                yaxis_range=yaxis_range
            )
            frame_paths.append(filename)

        video_path = os.path.join(output_dir, f"{seg_name}_pc{pc_index}_cineloop.mp4")
        create_video_from_frames(frame_paths, video_path, fps=fps)
        print(f"✅ Saved: {video_path}")


In [93]:
# def visualize_pca_segmental_(pca, mean_trace, segment_names, pc_index, output_dir="segmental_cineloops", fps=4):
#     os.makedirs(output_dir, exist_ok=True)
#     alphas = get_alpha_loop()
#
#     n_segments = len(segment_names)
#     segment_len = len(mean_trace) // n_segments
#
#     component_scaled = get_scaled_component(pca, pc_index)
#
#     yaxis_range = compute_yaxis_range(mean_trace, component_scaled, alphas)
#
#     for seg_idx, seg_name in enumerate(segment_names):
#         print(f"🎥 Segment: {seg_name} — PC {pc_index}")
#         start, end = seg_idx * segment_len, (seg_idx + 1) * segment_len
#         original_seg = mean_trace[start:end]
#         frame_dir = os.path.join(output_dir, f"frames_{seg_name}_pc{pc_index}")
#         os.makedirs(frame_dir, exist_ok=True)
#         frame_paths = []
#
#         for i, alpha in enumerate(alphas):
#             mod_trace = mean_trace + alpha * component_scaled
#             seg_trace = mod_trace[start:end]
#             label = f"α={alpha:.2f} × PC{pc_index}"
#             filename = os.path.join(frame_dir, f"{seg_name}_pc{pc_index}_frame_{i:03d}.png")
#
#             plot_trace_with_reference(
#                 trace=seg_trace,
#                 reference=original_seg,
#                 label=label,
#                 color=get_color(alpha),
#                 title=f"{seg_name} — PC {pc_index}",
#                 filename=filename,
#                 yaxis_range=yaxis_range
#             )
#
#             frame_paths.append(filename)
#
#         video_path = os.path.join(output_dir, f"{seg_name}_pc{pc_index}_cineloop.mp4")
#         create_video_from_frames(frame_paths, video_path, fps=fps)
#         print(f"✅ Saved: {video_path}")


# By the concatenation

In [94]:
def visualize_pca_concatenated(pca, mean_trace, segment_names, pc_index,
                               output_dir="concat_cineloops", fps=4, equalize_to_pc1=True):

    os.makedirs(output_dir, exist_ok=True)
    frame_dir = os.path.join(output_dir, f"frames_pc{pc_index}")
    os.makedirs(frame_dir, exist_ok=True)

    n_segments = len(segment_names)
    seg_len = len(mean_trace) // n_segments
    seg_bounds = [i * seg_len for i in range(n_segments + 1)]
    tickvals = [(seg_bounds[i] + seg_bounds[i+1]) // 2 for i in range(n_segments)]

    alphas = get_alpha_loop()

    # --- Build the PC effect in the reordered space ---
    pc_effect = effect_vector(pca, pc_index)
    pc_effect = reorder_vector_by_segments(pc_effect, segment_names, segment_names)  # <--- NOTE

    # Optional: equalize magnitude to PC1
    if equalize_to_pc1:
        pc1_effect = effect_vector(pca, 0)
        pc1_effect = reorder_vector_by_segments(pc1_effect, segment_names, segment_names)
        target_norm = np.linalg.norm(pc1_effect)
        pc_effect = rescale_to_target_norm(pc_effect, target_norm)

    yaxis_range = compute_yaxis_range(mean_trace, pc_effect, alphas)

    frame_paths = []
    for i, alpha in enumerate(alphas):
        mod_trace = mean_trace + alpha * pc_effect
        label = f"α={alpha:.2f} × PC{pc_index}"
        filename = os.path.join(frame_dir, f"pc{pc_index}_frame_{i:03d}.png")

        plot_trace_with_reference(
            trace=mod_trace,
            reference=mean_trace,
            label=label,
            color=get_color(alpha),
            title=f"PCA Component {pc_index} — {label}",
            filename=filename,
            yaxis_range=yaxis_range,
            x_ticks=tickvals,
            x_labels=segment_names,
            vlines=seg_bounds[1:-1]
        )
        frame_paths.append(filename)

    video_path = os.path.join(output_dir, f"cineloop_pc{pc_index}.mp4")
    create_video_from_frames(frame_paths, video_path, fps=fps)
    print(f"🎥 Saved: {video_path}")


In [95]:
# def visualize_pca_concatenated_(pca, mean_trace, segment_names, pc_index, output_dir="concat_cineloops", fps=4):
#     os.makedirs(output_dir, exist_ok=True)
#     frame_dir = os.path.join(output_dir, f"frames_pc{pc_index}")
#     os.makedirs(frame_dir, exist_ok=True)
#
#     n_segments = len(segment_names)
#     segment_len = len(mean_trace) // n_segments
#     segment_boundaries = [i * segment_len for i in range(n_segments + 1)]
#     tickvals = [(segment_boundaries[i] + segment_boundaries[i+1]) // 2 for i in range(n_segments)]
#
#     alphas = get_alpha_loop()
#     component_scaled = get_scaled_component(pca, pc_index)
#
#     yaxis_range = compute_yaxis_range(mean_trace, component_scaled, alphas)
#
#     frame_paths = []
#
#     for i, alpha in enumerate(alphas):
#         mod_trace = mean_trace + alpha * component_scaled
#         label = f"α={alpha:.2f} × PC{pc_index}"
#         filename = os.path.join(frame_dir, f"pc{pc_index}_frame_{i:03d}.png")
#
#         plot_trace_with_reference(
#             trace=mod_trace,
#             reference=mean_trace,
#             label=label,
#             color=get_color(alpha),
#             title=f"PCA Component {pc_index} — {label}",
#             filename=filename,
#             yaxis_range=yaxis_range,
#             x_ticks=tickvals,
#             x_labels=segment_names,
#             vlines=segment_boundaries[1:-1]
#         )
#
#         frame_paths.append(filename)
#
#     video_path = os.path.join(output_dir, f"cineloop_pc{pc_index}.mp4")
#     create_video_from_frames(frame_paths, video_path, fps=fps)
#     print(f"🎥 Saved: {video_path}")


In [96]:
for pc_idx in pcs_to_plot:

    visualize_pca_concatenated(pca, mean_trace_reordered, segment_names_reordered, pc_index=pc_idx)
    # visualize_pca_segmental(pca, mean_trace_reordered, segment_names_reordered, pc_index=pc_idx)

🎥 Saved: concat_cineloops\cineloop_pc13.mp4
🎥 Saved: concat_cineloops\cineloop_pc17.mp4
🎥 Saved: concat_cineloops\cineloop_pc32.mp4
